In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import joblib
import os

print("Libraries imported")

Libraries imported


In [2]:
df = pd.read_csv('../data/paysim.csv')
print(df.shape)
df.head()

(6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [3]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']


In [4]:
# nameOrig and nameDest are just IDs - not useful for model
# isFlaggedFraud is a rule based flag - we drop it to avoid leakage
df = df.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud'])
print(df.shape)
print(df.columns.tolist())

(6362620, 8)
['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'isFraud']


In [5]:
le = LabelEncoder()
df['type'] = le.fit_transform(df['type'])
print("Type encoding mapping:")
for i, cls in enumerate(le.classes_):
    print(f"  {cls} → {i}")

Type encoding mapping:
  CASH_IN → 0
  CASH_OUT → 1
  DEBIT → 2
  PAYMENT → 3
  TRANSFER → 4


In [6]:
# These features capture the KEY fraud pattern we found in EDA
# Fraud drains entire account balance

# How much of old balance was the transaction amount
df['balance_change_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']

# Did the sender balance go to zero after transaction
df['orig_balance_zero'] = (df['newbalanceOrig'] == 0).astype(int)

# Did the receiver balance not change even though money was sent
df['dest_balance_unchanged'] = (df['oldbalanceDest'] == df['newbalanceDest']).astype(int)

# Amount as ratio of old balance
df['amount_to_balance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1)

print("New features created")
print(df.shape)
df.head()

New features created
(6362620, 12)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,isFraud,balance_change_orig,orig_balance_zero,dest_balance_unchanged,amount_to_balance_ratio
0,1,3,9839.64,170136.0,160296.36,0.0,0.0,0,9839.64,0,1,0.057834
1,1,3,1864.28,21249.0,19384.72,0.0,0.0,0,1864.28,0,1,0.087731
2,1,4,181.00,181.0,0.00,0.0,0.0,1,181.00,1,1,0.994505
3,1,1,181.00,181.0,0.00,21182.0,0.0,1,181.00,1,0,0.994505
4,1,3,11668.14,41554.0,29885.86,0.0,0.0,0,11668.14,0,1,0.280788


In [7]:
print(df.dtypes)
print()
print(df.isnull().sum())

step                         int64
type                         int64
amount                     float64
oldbalanceOrg              float64
newbalanceOrig             float64
oldbalanceDest             float64
newbalanceDest             float64
isFraud                      int64
balance_change_orig        float64
orig_balance_zero            int64
dest_balance_unchanged       int64
amount_to_balance_ratio    float64
dtype: object

step                       0
type                       0
amount                     0
oldbalanceOrg              0
newbalanceOrig             0
oldbalanceDest             0
newbalanceDest             0
isFraud                    0
balance_change_orig        0
orig_balance_zero          0
dest_balance_unchanged     0
amount_to_balance_ratio    0
dtype: int64


In [8]:
X = df.drop('isFraud', axis=1)
y = df['isFraud']

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Fraud cases: {y.sum()}")
print(f"Genuine cases: {(y==0).sum()}")

X shape: (6362620, 11)
y shape: (6362620,)
Fraud cases: 8213
Genuine cases: 6354407


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"Fraud in train: {y_train.sum()}")
print(f"Fraud in test:  {y_test.sum()}")

X_train: (5090096, 11)
X_test:  (1272524, 11)
Fraud in train: 6570
Fraud in test:  1643


In [10]:
os.makedirs('../models', exist_ok=True)

joblib.dump(X_train, '../models/X_train.pkl')
joblib.dump(X_test,  '../models/X_test.pkl')
joblib.dump(y_train, '../models/y_train.pkl')
joblib.dump(y_test,  '../models/y_test.pkl')
joblib.dump(le,      '../models/label_encoder.pkl')

print("All files saved to models/ folder")

All files saved to models/ folder


In [11]:
import os
files = os.listdir('../models')
print("Files in models folder:")
for f in files:
    print(f"  {f}")

Files in models folder:
  label_encoder.pkl
  X_test.pkl
  X_train.pkl
  y_test.pkl
  y_train.pkl


In [12]:
print("=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print(f"Total features:     {X_train.shape[1]}")
print(f"Training samples:   {X_train.shape[0]:,}")
print(f"Test samples:       {X_test.shape[0]:,}")
print(f"Fraud in train:     {y_train.sum()}")
print(f"Fraud in test:      {y_test.sum()}")
print(f"Features: {X_train.columns.tolist()}")
print("=" * 50)
print("Ready for Notebook 03 - Model Training")

PREPROCESSING COMPLETE
Total features:     11
Training samples:   5,090,096
Test samples:       1,272,524
Fraud in train:     6570
Fraud in test:      1643
Features: ['step', 'type', 'amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'balance_change_orig', 'orig_balance_zero', 'dest_balance_unchanged', 'amount_to_balance_ratio']
Ready for Notebook 03 - Model Training
